# Lab 1 — จาก chatbot เป็น agent (EP03)

ในคลิปทฤษฎีเราเห็นว่า **agent** คือ LLM ที่ทำงานเป็นวง *รับรู้ → คิด → ลงมือ* ผ่าน **tool**
โดยมี **harness** คุมอยู่รอบนอก lab นี้เราจะสร้างผู้ช่วยของวิชา AI301 ที่ตอบได้ว่า
*"อีกกี่วันสอบกลางภาค"* และ **จองห้องติวให้จริง** — ด้วยโค้ดของเราเองทั้งหมด

**สิ่งที่จะได้เรียน**
1. chatbot: เรียก model ครั้งเดียว — ตอบไม่ได้ จองไม่ได้
2. tool: ฟังก์ชันปฏิทิน ดูห้องว่าง และจองห้อง + JSON schema ที่บอก model
3. agent loop: harness ตัวแรกของเรา (วนจนเสร็จ หรือจนต้องถามคน)
4. persistence: เก็บประวัติไว้ใน list จึงคุยต่อข้ามรอบได้
5. safety: จำกัดจำนวนรอบ + ให้คนอนุมัติก่อนจอง (human-in-the-loop)

ปฏิทินและห้องติวใน lab นี้เป็น **ข้อมูลจำลองที่เขียนขึ้นเพื่อการสอน** (วันนี้ = 10 พ.ย. 2026)
ส่วนคำตอบและการเรียก tool ของ model ทั้งหมดมาจากการรันจริง

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, ch15 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **6 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep03-lab1-agent-loop.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่องก่อนเริ่ม

| สิ่งที่ต้องมี | รายละเอียด |
|---|---|
| Python 3.10 ขึ้นไป | หรือใช้ Google Colab ซึ่งมีให้แล้ว |
| OpenRouter API key | สมัครที่ openrouter.ai → Keys → Create Key แล้วเติมเครดิต (lab นี้ใช้ไม่ถึง 10 สตางค์) |
| Packages | `openai`, `python-dotenv` (ติดตั้งใน cell ถัดไป) |

**ใส่ API key**
- **Colab:** ไอคอน 🔑 Secrets → ชื่อ `OPENROUTER_API_KEY` → เปิด *Notebook access*
- **เครื่องตัวเอง:** สร้างไฟล์ `.env` ในโฟลเดอร์เดียวกับ notebook แล้วใส่บรรทัด `OPENROUTER_API_KEY=sk-or-...`

⚠️ อย่าใส่ key ลงในโค้ดโดยตรง และอย่าแชร์ไฟล์ `.env`

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

## 1. เรียก Gemma ผ่าน OpenRouter

**ทำไมใช้ `google/gemma-4-26b-a4b-it`** — ทำตามคำสั่งภาษาไทยได้ดี และ **เรียก tool ได้** (tool calling)
ซึ่งเป็นหัวใจของ agent ใน lab นี้และ Lab 2

**ทำไมล็อก provider เป็น DeepInfra** — Parasail ที่ใช้ใน EP02 ไม่รองรับ tool เลย
เราทดสอบ agent ของ lab นี้กับ 10 provider ที่รองรับ tool (2026-11-10): ส่วนใหญ่เดินขั้นตอนเหมือนกันทุกครั้ง
แต่มีเจ้าหนึ่งตีความ "วันพฤหัสนี้" เป็นคนละวัน → เราล็อกไว้เจ้าเดียว (`deepinfra/fp8`) ให้ผลซ้ำได้

**ไม่ฟรี** แต่ถูกมาก: รันทั้ง lab หนึ่งรอบ ≈ 5,800 token ≈ 0.0006 ดอลลาร์ (ไม่ถึง 10 สตางค์)
· รุ่น `:free` ใช้ไม่ได้ (ทดสอบ 2026-11-10: 4 ใน 4 ครั้งได้ 429 "rate-limited upstream")

In [ ]:
import json, os
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())

In [ ]:
MODEL = "google/gemma-4-26b-a4b-it"
PROVIDER = {"order": ["deepinfra/fp8"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

## 2. แบบ chatbot: เรียก model ครั้งเดียว

ส่งคำขอของนักศึกษาให้ model ตรงๆ — ไม่มี tool ไม่มีลูป

In [ ]:
QUESTION = "อีกกี่วันจะสอบกลางภาควิชา AI301 แล้วช่วยจองห้องติววันพฤหัสนี้ช่วงบ่ายให้หน่อย"
reply = client.chat.completions.create(model=MODEL, extra_body=EXTRA,
                                       messages=[{"role": "user", "content": QUESTION}])
print(reply.choices[0].message.content)

## 3. Tool: ปฏิทินและระบบจองห้อง

ข้อมูลจำลองก่อน: ปฏิทินของวิชา (มีวันที่วันนี้ด้วย) และห้องติวที่ยังว่าง

In [ ]:
CALENDAR = {"today": "2026-11-10 (วันอังคาร)",
            "events": [{"name": "สอบกลางภาค AI301", "date": "2026-11-18 (วันพุธ)"},
                       {"name": "ส่งการบ้าน 3", "date": "2026-11-14 (วันเสาร์)"}]}
FREE = {"2026-11-12": {"ห้อง 1": ["09:00"], "ห้อง 3": ["13:00", "15:00"], "ห้อง 5": []},
        "2026-11-13": {"ห้อง 1": ["10:00", "13:00"], "ห้อง 3": [], "ห้อง 5": ["15:00"]}}
BOOKING = {}

In [ ]:
def get_calendar():
    return CALENDAR

def find_rooms(date):
    if date not in FREE:
        return {"error": f"ไม่มีข้อมูลห้องวันที่ {date} (มีข้อมูล: {', '.join(FREE)})"}
    return {"date": date, "free": {r: t for r, t in FREE[date].items() if t}}

# ✏️ TODO 1 (▶ วิดีโอ 5:53): ตรวจว่าเวลานั้นว่างจริงก่อนจอง (guardrail ในตัว tool)

def book_room(room, date, time):
    if ___:
        return {"ok": False, "error": f"{room} วันที่ {date} เวลา {time} ไม่ว่าง"}
    BOOKING.clear(); BOOKING.update(room=room, date=date, time=time)
    return {"ok": True, "booking": dict(BOOKING)}

**บอก model ว่ามี tool อะไรบ้าง** — แต่ละ tool คือ JSON schema: ชื่อ · ทำอะไร · รับ argument อะไร

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "get_calendar", "description": "ดูวันที่วันนี้ และปฏิทินสอบ/กำหนดส่งของวิชา AI301",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "find_rooms", "description": "ดูห้องติวที่ว่างในวันที่กำหนด",
        "parameters": {"type": "object", "required": ["date"],
                       "properties": {"date": {"type": "string", "description": "YYYY-MM-DD"}}}}},
    {"type": "function", "function": {
        "name": "book_room", "description": "จองห้องติว (จองได้ครั้งละหนึ่งห้อง ถ้ามีการจองเดิมจะถูกแทนที่)",
        "parameters": {"type": "object", "required": ["room", "date", "time"],
                       "properties": {"room": {"type": "string"}, "date": {"type": "string", "description": "YYYY-MM-DD"},
                                      "time": {"type": "string", "description": "HH:MM"}}}}},
]

In [ ]:
FUNCS = {"get_calendar": get_calendar, "find_rooms": find_rooms, "book_room": book_room}

# ✏️ TODO 2 (▶ วิดีโอ 7:51): แปลง arguments เป็น dict แล้วเรียกฟังก์ชันจริงตามชื่อที่ model ขอ

def run_tool(call):
    args = ___
    return json.dumps(___, ensure_ascii=False)

## 4. Agent loop = harness ตัวแรกของเรา

| ในลูป | จากคลิปทฤษฎี |
|---|---|
| ส่ง `history` + `TOOLS` ให้ model | 💭 คิด |
| model ขอ tool → `run_tool` | 🔧 ลงมือ |
| ผลของ tool กลับเข้า `history` | 👀 รับรู้ (observation) |
| ไม่ขอ tool แล้ว → ตอบ / ถามคน | 🏁 จบ / 🙋 ถามคน |
| `max_steps` | 🛑 guardrail: วนได้ไม่เกินกี่รอบ |

In [ ]:
SYSTEM = "คุณคือผู้ช่วยของวิชา AI301 ช่วยนักศึกษาดูปฏิทินและจองห้องติว ใช้ tool เพื่อหาข้อมูลจริง ห้ามเดาวันที่"
MAX_STEPS = 8
NEEDS_APPROVAL = set()   # ส่วนที่ 6 จะใส่ชื่อ tool ที่ต้องให้คนอนุมัติ
PENDING = []

In [ ]:
def call_llm(history):
    resp = client.chat.completions.create(model=MODEL, messages=history, tools=TOOLS, extra_body=EXTRA)
    return resp.choices[0].message

In [ ]:
def run_agent(history, max_steps=MAX_STEPS):
    for step in range(1, max_steps + 1):
        msg = call_llm(history)
        # ✏️ TODO 3 (▶ วิดีโอ 9:46): ทางออกของลูป: ถ้า model ไม่ขอ tool ให้เก็บคำตอบแล้ว return
        if ___:
            history.append({"role": "assistant", "content": msg.content})
            return msg.content
        history.append({"role": "assistant", "content": msg.content or "",
                        "tool_calls": [c.model_dump() for c in msg.tool_calls]})
        for call in msg.tool_calls:
            print(f"  🔧 {call.function.name}({call.function.arguments})")
            if call.function.name in NEEDS_APPROVAL:
                PENDING.append(call)
                continue
            # ✏️ TODO 4 (▶ วิดีโอ 10:35): ส่งผลของ tool กลับเข้าประวัติ (observation) พร้อม tool_call_id
            result = run_tool(call)
            print(f"  👀 {result}")
            history.append({"role": "tool", "tool_call_id": ___, "content": ___})
        if PENDING:
            return "⏸ รออนุมัติ"
    return f"⚠️ หยุด: ครบ {max_steps} รอบ"

In [ ]:
history = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}]
print(run_agent(history))

## 5. Persistence: ประวัติคือความจำของ agent

model ไม่จำอะไรข้ามการเรียก — สิ่งที่มันรู้คือสิ่งที่อยู่ใน `history` เท่านั้น

In [ ]:
# ✏️ TODO 5 (▶ วิดีโอ 12:20): คุยต่อในประวัติเดิม: เพิ่มข้อความของผู้ใช้ลงใน history ก่อนเรียก agent
history.___({"role": "user", "content": "13:00 ครับ"})
print(run_agent(history))
print(BOOKING)

In [ ]:
fresh = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "ขอเลื่อนเป็นบ่ายสามได้ไหม"}]
print(run_agent(fresh))

## 6. Safety: guardrail และ human-in-the-loop

ในส่วนที่ 5 agent **จองทันที** — ครั้งนี้ถูก แต่ถ้า model ตีความผิด การจองผิดคือผลจริง
ตำราบอกว่าการลงมือแบบอิสระต้องมี guardrail และ **ให้คนกำกับดูแล** → harness บังคับเอง ไม่ใช่หวังให้ model ถามเอง

| guardrail ใน lab นี้ | อยู่ที่ |
|---|---|
| วนได้ไม่เกิน `MAX_STEPS` รอบ | ลูป |
| จองได้เฉพาะเวลาที่ว่างจริง | `book_room` |
| `book_room` ต้องให้คนอนุมัติก่อน | `NEEDS_APPROVAL` + `approve()` |

In [ ]:
print(run_agent([{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}], max_steps=1))

In [ ]:
# ✏️ TODO 6 (▶ วิดีโอ 14:30): ใส่ชื่อ tool ที่ต้องให้คนอนุมัติก่อนรัน
NEEDS_APPROVAL = {___}

def approve(history, ok=True):
    while PENDING:
        call = PENDING.pop(0)
        result = run_tool(call) if ok else json.dumps({"ok": False, "error": "ผู้ใช้ไม่อนุมัติ"}, ensure_ascii=False)
        print(f"  ✅ {call.function.name} → {result}")
        history.append({"role": "tool", "tool_call_id": call.id, "content": result})
    return run_agent(history)

In [ ]:
history.append({"role": "user", "content": "ขอเลื่อนเป็นบ่ายสามได้ไหม"})
print(run_agent(history))
print(BOOKING)

In [ ]:
print(approve(history, ok=True))
print(BOOKING)

## สรุป: 5 ความท้าทาย ในโค้ดของเรา

| ความท้าทาย (ch15) | ใน lab นี้ |
|---|---|
| 🧠 Persistence | `history` list — คุยต่อได้ · ประวัติใหม่ = จำไม่ได้ |
| 📚 Grounding | `get_calendar` บอกวันที่วันนี้และวันสอบ → "อีก 8 วัน" |
| 🔧 Action | `find_rooms`, `book_room` + tool schema + `run_tool` |
| 👥 Coordination | — (agent ตัวเดียว · EP15–16) |
| 🛡️ Safety | `MAX_STEPS` · `book_room` ตรวจก่อนจอง · `NEEDS_APPROVAL` + `approve()` |

**ลองทำต่อ:** เพิ่ม tool `cancel_booking` แล้วให้ต้องอนุมัติด้วย ·
ลอง `approve(history, ok=False)` แล้วดูว่า agent ตอบอย่างไร

**คลิปถัดไป — Lab 2:** สร้างผู้ช่วยตัวเดิมด้วย LangChain (`@tool`, `create_agent`, checkpointer, human-in-the-loop)
แล้วเทียบกับโค้ดที่เราเขียนเองทีละชิ้น